# Chapter 5: Solving linear systems

Companion notebook for *Linear Algebra for Data Science and Machine Learning: Worked Examples and Python* by Dr. Sajjad Shaukat Jamal (Skills Outlet).

The cells below contain every Python example from Chapter 5, in the order they appear in the book. Work each example by hand first, then run the cell to check your result. Cells within a chapter build on each other, so run them from top to bottom.

## 5.4  Three equations and back substitution

### Check with Python

In [1]:
import numpy as np
A = np.array([[1.,1.,1.], [2.,3.,1.], [1.,-1.,2.]])
b = np.array([6.,11.,5.])
x = np.linalg.solve(A, b)
print(x)  # [1. 2. 3.]
print(b - A @ x)  # zero, up to rounding

[1. 2. 3.]
[0. 0. 0.]


For a square nonsingular system, solve computes the solution directly. The residual b − Ax measures how closely the computed vector satisfies the equations. Always interpret a residual relative to the scale and precision of the data.

## 5.8  Learn the elimination algorithm in Python

This educational implementation uses partial pivoting: at each step, select the remaining row with the largest absolute value in the pivot column. This avoids dividing by zero when a usable pivot exists and reduces some numerical difficulties. It does not fix an ill-conditioned problem.

In [2]:
def eliminate_solve(A, b, tol=1e-12):
    A = np.asarray(A, dtype=float).copy()
    b = np.asarray(b, dtype=float).copy()
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("Expected a square matrix")
    n = A.shape[0]
    if n == 0 or b.shape != (n,):
        raise ValueError("Expected a nonempty system and 1D b")
    scale = np.max(np.abs(A))
    for k in range(n):
        pivot = k + np.argmax(np.abs(A[k:, k]))
        if abs(A[pivot, k]) <= tol * scale:
            raise ValueError("Singular or numerically small pivot")
        A[[k, pivot]] = A[[pivot, k]]
        b[[k, pivot]] = b[[pivot, k]]
        for i in range(k + 1, n):
            factor = A[i, k] / A[k, k]
            A[i, k:] -= factor * A[k, k:]
            b[i] -= factor * b[k]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - A[i, i+1:] @ x[i+1:]) / A[i, i]
    return x

### Compare with the library solver

In [3]:
A = np.array([[1.,1.,1.], [2.,3.,1.], [1.,-1.,2.]])
b = np.array([6.,11.,5.])
x = eliminate_solve(A, b)
assert np.allclose(x, np.linalg.solve(A, b))
assert np.allclose(A @ x, b)
print(x)  # [1. 2. 3.]

[1. 2. 3.]


The tolerance is a teaching choice scaled by the largest input coefficient. This function is limited to finite, square systems and does not classify singular systems or provide a condition estimate. Use a tested numerical library for substantial work. Try it on a system that needs a row swap, such as Example 3, and on a singular system to see the error it raises.

## 5.9  An application to model fitting

### A third observation may conflict

Least squares chooses parameters that minimize the sum of squared residuals. We will derive that method in Chapter 8. Here we preview its numerical result so that the reason for studying approximate solutions is concrete.

In [4]:
X = np.array([[1.,1.], [1.,3.], [1.,2.]])
y = np.array([50.,70.,63.])
beta, _, rank, _ = np.linalg.lstsq(X, y, rcond=None)
pred = X @ beta
residual = y - pred
print(beta)      # [41. 10.]
print(residual)  # [-1. -1.  2.]
print(round(residual @ residual, 6))  # 6.0

[41. 10.]
[-1. -1.  2.]
6.0


The approximate line predicts (51, 71, 61). Its residuals sum to zero here because the design includes an intercept. Its squared residual total is 1 + 1 + 4 = 6. The exact line through the first two observations has residuals (0, 0, 3), with squared total 9.

## 5.10  Determinants and invertibility

### A determinant does not measure sensitivity

The nearly duplicate system of Example 10 has coefficient rows (1, 1) and (1, 1.001), with determinant 0.001. Multiplying every equation—including its right-hand side—by 10 leaves the solution set unchanged and multiplies the determinant of this 2 × 2 coefficient matrix by 100. Its relative condition number is unchanged. A small determinant is therefore not a reliable warning sign, and det(A) == 0 is not a safe numerical test. Use rank or the condition number, developed with singular values in Chapter 10.

In [5]:
import numpy as np
A = np.array([[1., 1., 1.], [2., 3., 1.], [1., -1., 2.]])
print(round(np.linalg.det(A), 6))  # -1.0
B = np.array([[1., 2.], [2., 4.]])
print(np.linalg.det(B))            # 0.0
print(np.linalg.matrix_rank(B))    # 1

-1.0
0.0
1


Chapter 9 uses determinants in one essential way: λ is an eigenvalue of A exactly when A − λI is singular, that is, when det(A − λI) = 0.

## 5.14  Optional extension: Cholesky and numerical checks

### Example 17  Use symmetric positive definite structure

For b = (6, 5), solve Lz = b to obtain z = (3, √2). Then solve Lᵀx = z: √2x₂ = √2 and 2x₁ + x₂ = 3, giving x = (1, 1). A singular positive semidefinite matrix does not have this nonsingular Cholesky factor; do not apply the same solver without checking the required conditions.

In [6]:
import numpy as np
A = np.array([[4., 2.], [2., 3.]])
b = np.array([6., 5.])
L = np.linalg.cholesky(A)
z = np.linalg.solve(L, b)
x = np.linalg.solve(L.T, z)
assert np.allclose(L @ L.T, A)
assert np.allclose(A @ x, b)
B = np.array([[1., 1.], [1., 0.], [0., 1.]])
Q, R = np.linalg.qr(B, mode='reduced')
assert np.allclose(Q.T @ Q, np.eye(2))
assert np.allclose(Q @ R, B)